# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('/content/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [2]:
# Task 1
# Fossil sources only
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

# CVD-safe palette from the course (blue / orange / yellow), one colour per fossil source
fossil_colors = {
    'Coal':        '#E07B39',   # orange
    'Oil':         '#2E75B6',   # blue
    'Natural Gas': '#F5C842',   # yellow
}
parent_grey = '#DDDDDD'

# Numbers for the insight title
region_tot = df_fossil.groupby('Region')['TWh'].sum().sort_values(ascending=False)
top_region, top_region_twh = region_tot.index[0], region_tot.iloc[0]
top_share = top_region_twh / region_tot.sum()

fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=fossil_colors,
    labels={'Source': 'Fossil source', 'TWh': 'Energy (TWh)'},
    height=700, width=1100,
    template='plotly_white',
)

fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',          # TWh only, no percentages
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
    root_color='white',
)

# Grey out parent nodes (Region and Country level): keep palette colours only on the leaves
fig.data[0].marker.colors = [
    c if c in fossil_colors.values() else parent_grey
    for c in fig.data[0].marker.colors
]

fig.update_layout(
    title=(f'<b>{top_region}</b> is the most fossil-dependent region: '
           f'{top_region_twh:,.0f} TWh, {top_share:.0%} of all fossil energy in the dataset'),
    font=dict(family='Arial', size=12),
    margin=dict(t=80, l=10, r=10, b=10),
)
fig.show()

## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [3]:
# Task 2 - charts
# Aggregate low-carbon (Nuclear + Hydro) TWh by country
df_low = (df.loc[df['Source_Type'] == 'Low-carbon']
            .groupby('Country', as_index=False)['TWh'].sum()
            .sort_values('TWh', ascending=False))
df_low['All'] = 'Low-carbon'          # dummy root node for the treemap

BLUE = '#2E75B6'
leader, leader_twh = df_low.iloc[0]['Country'], df_low.iloc[0]['TWh']

# ── Treemap ──
fig_tm = px.treemap(
    df_low, path=['All', 'Country'], values='TWh',
    labels={'TWh': 'Energy (TWh)'},
    height=550, width=1000, template='plotly_white',
)
fig_tm.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
    root_color='white',
)
# leaves blue, root (dummy parent) grey
fig_tm.data[0].marker.colors = [BLUE if '/' in i else '#DDDDDD' for i in fig_tm.data[0].ids]
fig_tm.update_layout(title='Low-carbon energy (Nuclear + Hydro) by country, TWh',
                     font=dict(family='Arial', size=12), margin=dict(t=70, l=10, r=10, b=10))
fig_tm.show()

# ── Horizontal bar ──
fig_bar = px.bar(
    df_low, x='TWh', y='Country', orientation='h',
    text='TWh',
    color_discrete_sequence=[BLUE],
    category_orders={'Country': df_low['Country'].tolist()},   # highest TWh first
    labels={'TWh': 'Low-carbon energy (TWh)'},
    height=550, width=1000, template='plotly_white',
)
fig_bar.update_traces(texttemplate='%{x:,.0f} TWh', textposition='outside', cliponaxis=False)
fig_bar.update_layout(
    title=f'<b>{leader}</b> leads low-carbon energy with {leader_twh:,.0f} TWh',
    font=dict(family='Arial', size=12),
    yaxis=dict(title='', categoryorder='array', categoryarray=df_low['Country'].tolist(), autorange='reversed'),
    xaxis=dict(visible=False),
    margin=dict(t=70, l=10, r=80, b=10),
)
fig_bar.show()

### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

The **bar chart** tells the ranking story more clearly. All bars share one baseline and are sorted, so the reader compares *length*, which is easy to judge, and the order from France down to Australia is read straight off the axis. In the treemap the reader has to compare *area* of rectangles with different shapes, which is much harder. Countries with similar totals (for example Canada and Norway, or Germany, the UK and Japan) are almost impossible to rank by eye, and the ranking is not given explicitly because the layout only roughly follows size. The treemap is still useful for showing how big each country is relative to the whole, but when the message is "who is first, and by how much", use the bar chart.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [4]:
# Task 3
# Reuse df_fossil from Task 1: aggregate Region x Source and convert to shares of each region's fossil total
reg = df_fossil.groupby(['Region', 'Source'], as_index=False)['TWh'].sum()

# Some region/source pairs have no rows (e.g. Middle East has no coal): fill them with 0 so every region has 3 segments
full = pd.MultiIndex.from_product([reg['Region'].unique(), ['Coal', 'Oil', 'Natural Gas']], names=['Region', 'Source'])
reg = reg.set_index(['Region', 'Source']).reindex(full, fill_value=0).reset_index()

reg['Share'] = reg['TWh'] / reg.groupby('Region')['TWh'].transform('sum')

# Sort regions by coal share, highest at the top
region_order = (reg.loc[reg['Source'] == 'Coal']
                   .sort_values('Share', ascending=False)['Region'].tolist())
top_coal_region = region_order[0]
top_coal_share = reg.loc[(reg['Region'] == top_coal_region) & (reg['Source'] == 'Coal'), 'Share'].iloc[0]

# Labels on the coal segments only (skip empty segments)
reg['Label'] = np.where((reg['Source'] == 'Coal') & (reg['Share'] > 0), reg['Share'].map('{:.0%}'.format), '')

# Coal keeps its Task 1 colour (orange); Oil and Natural Gas are greys
accent_mapping = {
    'Coal':        fossil_colors['Coal'],   # '#E07B39', same as Task 1
    'Oil':         '#BBBBBB',
    'Natural Gas': '#DDDDDD',
}

fig = px.bar(
    reg, x='Share', y='Region', color='Source', text='Label',
    color_discrete_map=accent_mapping,
    category_orders={'Region': region_order,
                     'Source': ['Coal', 'Oil', 'Natural Gas']},   # coal on the baseline
    labels={'Share': 'Share of fossil energy', 'Source': 'Fossil source'},
    hover_name='Region', hover_data={'Label': False, 'Share': ':.0%'},
    height=550, width=1000, template='plotly_white',
)
fig.update_traces(
    textposition='inside', insidetextanchor='start',
    textfont=dict(color='white', size=12),
    marker_line_color='white', marker_line_width=1,
)
fig.update_layout(
    title=dict(text=f'<b>{top_coal_region}</b> relies most on coal: {top_coal_share:.0%} of its fossil energy'),
    font=dict(family='Arial', size=12),
    bargap=0.25,
    legend=dict(orientation='h', x=0, y=1.08),
    xaxis=dict(visible=False, range=[0, 1]),
    yaxis=dict(title='', categoryorder='array', categoryarray=region_order, autorange='reversed'),
)
fig.add_annotation(x=1, y=0, xref='x', yref='paper', xanchor='right', yanchor='top',
                   yshift=-4, text='100%', showarrow=False, font=dict(size=11, color='grey'))
fig.show()

### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

- **Treemap:** *Where is the most fossil energy consumed in absolute terms, and which countries make up each region's total?* The areas show magnitude and the Region → Country → Source nesting, e.g. Asia dominates, with China and India as its biggest parts. The stacked bar only shows shares, so it hides the size of each region (a small region and a huge one look equally long) and it has no country level.
- **100% stacked bar:** *Which region depends most on coal relative to its own fossil mix, and by how much compared with the others?* Coal sits on a common baseline, so the coal shares can be compared precisely (Africa vs Oceania vs Asia and so on). In the treemap, comparing proportions across regions means comparing areas of differently sized blocks, which is not accurate.